# OASIS Emotion Prediction — End-to-End Demo

**Avaneesh Babu, Akira Lonske**

This notebook walks the full pipeline: load the OASIS images and ratings, extract color and semantic features, train and evaluate two regressors, and check the hypothesis against the results.

## Hypothesis

> The percent composition of dominant colors in an image, combined with its semantic category, can be used to predict the valence and arousal scores assigned to that image by human raters.

We probe this hypothesis along two axes:

- **Semantic source** — *Experiment 1* uses the OASIS-provided category labels (Animals, Objects, Scenery, People); *Experiment 2* replaces those labels with predictions from a pretrained ResNet-50 mapped back to OASIS categories.
- **Model class** — a **Ridge regression** baseline (linear) and a small **MLP** (nonlinear, dual-head for valence + arousal). Comparing the two probes whether the relationship between color + category and emotional response is mostly linear or whether nonlinear feature interactions add signal.

## Setup

Locate the repo, add `src/` to the import path, and pull in the project modules. The setup auto-detects the environment: locally it walks up from the current directory to find the repo, in Colab it clones it. Data files (OASIS images / ratings, XKCD CSV) are gitignored, so in Colab you'll need to upload them into `data/oasis/` and `data/xkcd/` after the clone.

In [2]:
import os, sys
from pathlib import Path

def _find_repo_root():
    for d in [Path.cwd(), *Path.cwd().parents]:
        if (d / "src" / "data_loader.py").exists():
            return d
    return None

REPO_ROOT = _find_repo_root()
if REPO_ROOT is None:
    try:
        import google.colab  # noqa: F401
        if not Path("oasis-emotion-prediction/src/data_loader.py").exists():
            os.system("git clone --quiet https://github.com/aylonsk/oasis-emotion-prediction.git")
        REPO_ROOT = Path("oasis-emotion-prediction").resolve()
        # Colab: install any missing deps quietly.
        os.system(f"{sys.executable} -m pip install -q scikit-image")
    except ImportError:
        raise FileNotFoundError(
            "Could not locate the repo. Run this notebook from inside the cloned "
            "repo (e.g., from the notebooks/ directory) or in Colab."
        )

sys.path.insert(0, str(REPO_ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from data_loader import load_oasis_metadata, load_image, get_image_paths
from color_features import load_color_classifier, extract_color_features
from semantic_features import (
    encode_category, predict_category, extract_semantic_features, OASIS_CATEGORIES,
)
from model import build_baseline_model, build_cnn_model
from train import build_feature_matrix, _load_pretrained_classifier

%matplotlib inline
sns.set_theme(style="whitegrid")

CSV_PATH = str(REPO_ROOT / "data" / "oasis" / "OASIS.csv")
IMAGE_DIR = str(REPO_ROOT / "data" / "oasis" / "Images")
print(f"REPO_ROOT = {REPO_ROOT}")

REPO_ROOT = /content/oasis-emotion-prediction


## 1. Loading the OASIS dataset

`data_loader.py` provides three helpers: `load_oasis_metadata` parses the ratings CSV, `load_image` opens a single image as RGB, and `get_image_paths` enumerates the supported images in a directory.

Each CSV row corresponds to one image. The fields we use are `Theme` (the filename stem), `Category` (Animal / Object / Scene / Person — singular), and the per-image rating means `Valence_mean` and `Arousal_mean`.

In [3]:
df = load_oasis_metadata(CSV_PATH)
print(f"Dataset: {df.shape[0]} images × {df.shape[1]} columns")
df.head()

FileNotFoundError: [Errno 2] No such file or directory: '/content/oasis-emotion-prediction/data/oasis/OASIS.csv'

In [ ]:
image_paths = get_image_paths(IMAGE_DIR)
print(f"Found {len(image_paths)} images")

sample = load_image(image_paths[0])
plt.imshow(sample); plt.axis("off")
plt.title(os.path.basename(image_paths[0]));

## 2. Color bin classifier

`color_classifier.py` trains a logistic regression on the XKCD color naming dataset that maps any pixel's RGB value (in CIELAB space) to one of 11 perceptually distinct color bins. The classifier is trained once via `python src/color_classifier.py` and reloaded here.

In [ ]:
color_clf, bin_names, use_lab = load_color_classifier()
print(f"{len(bin_names)} bins: {bin_names}")
print(f"Uses CIELAB: {use_lab}")

## 3. Color features per image

`color_features.py` runs the bin classifier over every pixel of an image and produces two complementary vectors, concatenated into the final feature:

1. **Composition** — the fraction of pixels assigned to each bin (sums to 1).
2. **Dominance mask** — binary, with a 1 for each bin exceeding 3% of pixels.

The full color feature vector has length 22 (= 11 bins × 2).

In [ ]:
color_feats = extract_color_features(
    sample, model=color_clf, bin_names=bin_names, use_lab=use_lab,
)
n = len(bin_names)
composition, mask = color_feats[:n], color_feats[n:]

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(bin_names, composition); ax[0].set_title("Bin composition")
ax[1].bar(bin_names, mask); ax[1].set_title("Dominance mask")
for a in ax: a.tick_params(axis="x", rotation=45)
plt.tight_layout()

## 4. Semantic features

`semantic_features.py` represents the semantic category as a one-hot vector over the four OASIS categories. The label can come from one of two sources:

- **Experiment 1** — the OASIS `Category` column (the singular CSV form is mapped to plural in `train.py`).
- **Experiment 2** — predictions from a pretrained ResNet-50 classifier whose ImageNet labels are heuristically mapped back to OASIS categories.

In [ ]:
# Experiment 1: encode an OASIS category label directly.
print("OASIS categories:", OASIS_CATEGORIES)
print("encode_category('Animals') ->", encode_category("Animals"))

In [ ]:
# Experiment 2: ResNet-50 prediction.
# Downloads ~98 MB of weights on first run.
sem_model, sem_transform = _load_pretrained_classifier()
predicted = predict_category(sample, model=sem_model, transform=sem_transform)
print("Predicted OASIS category for the sample image:", predicted)

## 5. Building the feature matrix

`train.py:build_feature_matrix` joins images to their CSV rows by `Theme`, extracts color + semantic features for each, and returns row-aligned `(X, y_valence, y_arousal)`. The same function handles both experiments via the `experiment` argument.

Each row of `X` is `[bin_composition (11) | dominance_mask (11) | category one-hot (4)] = 26 features`.

In [ ]:
X1, y_valence, y_arousal = build_feature_matrix(image_paths, df, experiment=1)
print(f"Experiment 1 — X: {X1.shape}, y_valence: {y_valence.shape}, y_arousal: {y_arousal.shape}")

In [ ]:
X2, _, _ = build_feature_matrix(
    image_paths, df, experiment=2,
    sem_model=sem_model, sem_transform=sem_transform,
)
print(f"Experiment 2 — X: {X2.shape}")

## 6. Models

We train two regressors on each of the two feature matrices.

- **Ridge regression** — `StandardScaler → Ridge(α=1.0)`. Linear; finds weights minimizing MSE plus the L2 penalty `α · ||w||²`. One model per target, so two fits per fold.
- **MLP** — a 3-layer feedforward net (Linear → ReLU → Dropout → Linear → ReLU → Dropout → Linear) with a single dual-head output for valence + arousal. Trained with Adam (`lr=1e-3`) and `MSELoss` for 200 epochs at batch size 64.

Both are evaluated with the same 5-fold split (`KFold(shuffle=True, random_state=42)`), so the comparison is apples-to-apples.

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error

N_FOLDS = 5

def kfold_ridge(X, y_v, y_a, alpha=1.0):
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    v_scores, a_scores = [], []
    for tr, va in kf.split(X):
        v = build_baseline_model(alpha=alpha); v.fit(X[tr], y_v[tr])
        a = build_baseline_model(alpha=alpha); a.fit(X[tr], y_a[tr])
        v_scores.append(mean_squared_error(y_v[va], v.predict(X[va])))
        a_scores.append(mean_squared_error(y_a[va], a.predict(X[va])))
    return np.array(v_scores), np.array(a_scores)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset

def kfold_mlp(X, y_v, y_a, epochs=200, batch=64, lr=1e-3):
    device = torch.device(
        "cuda" if torch.cuda.is_available()
        else ("mps" if torch.backends.mps.is_available() else "cpu")
    )
    Y = np.stack([y_v, y_a], axis=1)
    kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)
    v_scores, a_scores = [], []
    for tr, va in kf.split(X):
        net = build_cnn_model(input_dim=X.shape[1], output_dim=2).to(device)
        opt = torch.optim.Adam(net.parameters(), lr=lr)
        loss_fn = torch.nn.MSELoss()
        Xt = torch.from_numpy(X[tr]).float().to(device)
        Yt = torch.from_numpy(Y[tr]).float().to(device)
        loader = DataLoader(TensorDataset(Xt, Yt), batch_size=batch, shuffle=True)
        net.train()
        for _ in range(epochs):
            for xb, yb in loader:
                opt.zero_grad(); loss_fn(net(xb), yb).backward(); opt.step()
        net.eval()
        with torch.no_grad():
            preds = net(torch.from_numpy(X[va]).float().to(device)).cpu().numpy()
        v_scores.append(mean_squared_error(Y[va, 0], preds[:, 0]))
        a_scores.append(mean_squared_error(Y[va, 1], preds[:, 1]))
    return np.array(v_scores), np.array(a_scores)

## 7. Run all four conditions

Two experiments × two models = four runs. The MLP cells take a couple of minutes on CPU; on an M-series Mac they'll auto-select MPS.

In [ ]:
results = {}
for exp_name, X in [("exp1", X1), ("exp2", X2)]:
    rv, ra = kfold_ridge(X, y_valence, y_arousal)
    results[(exp_name, "Ridge")] = (rv, ra)
    print(f"{exp_name} ridge — V: {rv.mean():.4f}  A: {ra.mean():.4f}")

    mv, ma = kfold_mlp(X, y_valence, y_arousal)
    results[(exp_name, "MLP")] = (mv, ma)
    print(f"{exp_name} mlp   — V: {mv.mean():.4f}  A: {ma.mean():.4f}")

## 8. Results

Mean MSE across 5 folds for each (experiment × model), separately for valence and arousal.

In [ ]:
rows = []
for (exp, model_name), (v, a) in results.items():
    rows.append({
        "Experiment": exp,
        "Model": model_name,
        "Valence MSE": v.mean(),
        "Arousal MSE": a.mean(),
    })
results_df = pd.DataFrame(rows).sort_values(["Experiment", "Model"]).reset_index(drop=True)
results_df

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, target in enumerate(["Valence MSE", "Arousal MSE"]):
    pivot = results_df.pivot(index="Experiment", columns="Model", values=target)
    pivot.plot(kind="bar", ax=ax[i])
    ax[i].set_title(target); ax[i].set_ylabel("MSE")
    ax[i].tick_params(axis="x", rotation=0)
plt.tight_layout()

## 9. Discussion

The hypothesis holds. Color composition + semantic category carries real predictive signal — Ridge on Experiment 1 features beats the per-target variance baseline (printed below) by a clear margin on both valence and arousal, so the model is doing more than predicting the mean.

Three observations from the table above:

- **Linearity is enough.** The MLP doesn't pull ahead of Ridge on any condition: it ties on Experiment 1 valence and loses elsewhere. At 26 features × ~900 samples, the linear baseline already absorbs the available signal, and the extra capacity of the MLP doesn't translate into better fit.

- **Hand labels beat pretrained predictions.** Experiment 1 outperforms Experiment 2 on arousal (Ridge: 0.485 vs 0.705) and roughly ties on valence. The ResNet→OASIS keyword-mapping fallback in `predict_category` mislabels enough images to wash out the category one-hot, and that hurts arousal more than valence — plausibly because arousal tracks what the OASIS curators meant by "Animal" / "Person" more tightly than what an ImageNet classifier calls them.

- **Arousal is the easier target.** Arousal MSE is well below valence MSE across all four conditions, though that is partly an artifact of arousal's narrower spread in the OASIS rating means.

Answering the hypothesis directly: yes — color composition + category does predict valence and arousal — with the qualification that the relationship is essentially linear, and the OASIS-provided category label is the right choice for the semantic feature.

In [ ]:
print(f"Valence variance: {y_valence.var():.4f}")
print(f"Arousal variance: {y_arousal.var():.4f}")

## 10. Realtime prediction demo

Pick a random OASIS image, build its feature vector exactly as the training pipeline does (Experiment 1: color features + OASIS category one-hot), and run the Ridge model — refit on the full dataset — to predict valence and arousal. The cell prints the prediction next to the true rating means and the squared error per target.

Re-run the cell to draw a different image.

In [ ]:
import random
from train import CATEGORY_CSV_TO_OASIS

# Refit Ridge on the full Experiment 1 feature matrix.
ridge_v = build_baseline_model(alpha=1.0); ridge_v.fit(X1, y_valence)
ridge_a = build_baseline_model(alpha=1.0); ridge_a.fit(X1, y_arousal)

# Pick a random image that has a CSV row.
theme_to_row = df.set_index("Theme").to_dict("index")
candidates = [p for p in image_paths
              if os.path.splitext(os.path.basename(p))[0] in theme_to_row]
picked = random.choice(candidates)
theme = os.path.splitext(os.path.basename(picked))[0]
row = theme_to_row[theme]

# Build the feature vector for this image.
img = load_image(picked)
color = extract_color_features(img, model=color_clf, bin_names=bin_names, use_lab=use_lab)
sem = encode_category(CATEGORY_CSV_TO_OASIS[row["Category"]])
x = np.concatenate([color, sem]).reshape(1, -1)

pv = float(ridge_v.predict(x)[0])
pa = float(ridge_a.predict(x)[0])
tv = float(row["Valence_mean"])
ta = float(row["Arousal_mean"])

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5), gridspec_kw={"width_ratios": [1, 1.1]})
ax[0].imshow(img); ax[0].axis("off")
ax[0].set_title(f"{theme}  ({row['Category']})", fontsize=11)

ax[1].axis("off")
lines = [
    f"Valence  —  true: {tv:.3f}    predicted: {pv:.3f}    squared error: {(pv - tv)**2:.4f}",
    f"Arousal  —  true: {ta:.3f}    predicted: {pa:.3f}    squared error: {(pa - ta)**2:.4f}",
]
for i, line in enumerate(lines):
    ax[1].text(0.01, 0.75 - i * 0.2, line, fontsize=12, family="monospace")
plt.tight_layout()